# Submission

Locked-in solution: LightGBM (multiclass, balanced class weights) + StratifiedKFold k=5 + post-hoc per-class log-offset calibration tuned on OOF probabilities. All 19 features. Full 630k rows.

## Config

In [1]:
DATA_DIR        = "../data"
SUBMISSION_PATH = "../submissions/submission.csv"

SEED     = 42
N_SPLITS = 5

TARGET = "Irrigation_Need"
ID_COL = "id"

TARGET_MAP     = {"Low": 0, "Medium": 1, "High": 2}
INV_TARGET_MAP = {v: k for k, v in TARGET_MAP.items()}
CLASSES        = ["Low", "Medium", "High"]

## Imports

In [2]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

import lightgbm as lgb

warnings.filterwarnings("ignore")
np.random.seed(SEED)

## Load data

In [3]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"train: {train.shape}")
print(f"test : {test.shape}")

train: (630000, 21)
test : (270000, 20)


## Prepare features

In [4]:
features = [c for c in train.columns if c not in (ID_COL, TARGET)]
cat_cols = train[features].select_dtypes(include="object").columns.tolist()

y      = train[TARGET].map(TARGET_MAP).values
X      = train[features].copy()
X_test = test[features].copy()

for c in cat_cols:
    X[c]      = X[c].astype("category")
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

print(f"X: {X.shape}, X_test: {X_test.shape}")
print(f"categorical: {cat_cols}")

X: (630000, 19), X_test: (270000, 19)
categorical: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


## Train — 5-fold stratified CV

In [5]:
LGB_PARAMS = dict(
    objective        = "multiclass",
    num_class        = 3,
    metric           = "multi_logloss",
    learning_rate    = 0.05,
    num_leaves       = 63,
    min_data_in_leaf = 200,
    feature_fraction = 0.9,
    bagging_fraction = 0.9,
    bagging_freq     = 5,
    lambda_l2        = 1.0,
    verbose          = -1,
    seed             = SEED,
    class_weight     = "balanced",
)
NUM_BOOST_ROUND = 2000
EARLY_STOP      = 100

In [6]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_proba   = np.zeros((len(X), 3))
test_proba  = np.zeros((len(X_test), 3))
fold_scores = []
models      = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    dtrain = lgb.Dataset(X_tr, y_tr, categorical_feature=cat_cols)
    dvalid = lgb.Dataset(X_val, y_val, categorical_feature=cat_cols, reference=dtrain)

    model = lgb.train(
        LGB_PARAMS, dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dvalid],
        callbacks=[
            lgb.early_stopping(EARLY_STOP, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    oof_proba[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_proba += model.predict(X_test, num_iteration=model.best_iteration) / N_SPLITS

    score = balanced_accuracy_score(y_val, oof_proba[val_idx].argmax(axis=1))
    fold_scores.append(score)
    models.append(model)
    print(f"fold {fold+1}: best_iter={model.best_iteration:>4d}  BalAcc={score:.5f}")

print(f"\nmean fold BalAcc = {np.mean(fold_scores):.5f}  std = {np.std(fold_scores):.5f}")

fold 1: best_iter= 519  BalAcc=0.96120
fold 2: best_iter= 511  BalAcc=0.96296
fold 3: best_iter= 405  BalAcc=0.96261
fold 4: best_iter= 445  BalAcc=0.96069
fold 5: best_iter= 538  BalAcc=0.96196

mean fold BalAcc = 0.96188  std = 0.00085


## Post-hoc threshold tuning — per-class log-offsets on OOF

In [7]:
log_oof = np.log(oof_proba + 1e-12)
grid    = np.round(np.arange(-1.2, 0.41, 0.1), 2)

best_offsets, best_score = (0.0, 0.0, 0.0), -np.inf
for o_low in grid:
    for o_med in grid:
        sc = balanced_accuracy_score(
            y, (log_oof + np.array([o_low, o_med, 0.0])).argmax(axis=1))
        if sc > best_score:
            best_offsets, best_score = (float(o_low), float(o_med), 0.0), float(sc)

oof_balacc_raw = balanced_accuracy_score(y, oof_proba.argmax(axis=1))
print(f"OOF BalAcc (raw argmax) : {oof_balacc_raw:.5f}")
print(f"OOF BalAcc (tuned)      : {best_score:.5f}")
print(f"best offsets            : {best_offsets}")
print(f"lift                    : +{best_score - oof_balacc_raw:.5f}")

OOF BalAcc (raw argmax) : 0.96188
OOF BalAcc (tuned)      : 0.96712
best offsets            : (-1.2, -1.2, 0.0)
lift                    : +0.00524


## Submission

In [8]:
def calibrate(proba):
    return (np.log(proba + 1e-12) + np.array(best_offsets)).argmax(axis=1)

test_int = calibrate(test_proba)
test_lbl = pd.Series(test_int).map(INV_TARGET_MAP).values

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_lbl})

os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"saved \u2192 {SUBMISSION_PATH}  ({len(submission):,} rows)")

print("\npredicted distribution:")
print(submission[TARGET].value_counts(normalize=True).round(4))
print("\ntrain prior:")
print(train[TARGET].value_counts(normalize=True).round(4))

saved → ../submissions/submission.csv  (270,000 rows)

predicted distribution:
Irrigation_Need
Low       0.5920
Medium    0.3745
High      0.0334
Name: proportion, dtype: float64

train prior:
Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64


## Evaluation

In [9]:
oof_pred_tuned = calibrate(oof_proba)

print(f"OOF BalAcc (raw argmax) : {oof_balacc_raw:.5f}")
print(f"OOF BalAcc (tuned)      : {balanced_accuracy_score(y, oof_pred_tuned):.5f}")
print(f"per-fold BalAcc         : {[round(s, 5) for s in fold_scores]}")
print(f"mean / std              : {np.mean(fold_scores):.5f} / {np.std(fold_scores):.5f}")

OOF BalAcc (raw argmax) : 0.96188
OOF BalAcc (tuned)      : 0.96712
per-fold BalAcc         : [np.float64(0.9612), np.float64(0.96296), np.float64(0.96261), np.float64(0.96069), np.float64(0.96196)]
mean / std              : 0.96188 / 0.00085


In [10]:
print("Classification report (OOF, tuned):\n")
print(classification_report(y, oof_pred_tuned, target_names=CLASSES, digits=4))

Classification report (OOF, tuned):

              precision    recall  f1-score   support

         Low     0.9854    0.9948    0.9901    369917
      Medium     0.9860    0.9719    0.9789    239074
        High     0.9392    0.9347    0.9369     21009

    accuracy                         0.9841    630000
   macro avg     0.9702    0.9671    0.9686    630000
weighted avg     0.9841    0.9841    0.9841    630000



In [11]:
cm = confusion_matrix(y, oof_pred_tuned)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES],
                          columns=[f"pred_{c}" for c in CLASSES])
print("Confusion matrix (OOF, tuned):\n")
print(cm_df)

Confusion matrix (OOF, tuned):

             pred_Low  pred_Medium  pred_High
true_Low       367991         1925          1
true_Medium      5453       232350       1271
true_High           0         1372      19637
